# 🎓 M.A.C.H. AI — Academic Tool Router QLoRA Fine-Tuning
### Multimodal Academic Cognitive Hub · Amrita Vishwa Vidyapeetham

This Google Colab notebook fine-tunes a lightweight LLM using **QLoRA (4-bit NF4 Quantization + LoRA)** on the **M.A.C.H. AI Curated Academic Router Dataset**.

---

### ⚡ Highlights & Benchmarks:
* **Runtime**: ~4–10 minutes total on a free Google Colab **T4 GPU**.
* **Dataset**: 271 curated examples (202 training / 69 held-out evaluation) covering 9 registered academic tools + conversational fallback.
* **Base Model**: `Qwen/Qwen2.5-3B-Instruct` (fast, ungated, high-precision function calling) or `google/gemma-2-2b-it` / `google/gemma-2-9b-it`.
* **Objective**: Teaches the model to select the exact campus tool and extract only unambiguous parameters without hallucinating facts.

### 📌 Prerequisites:
Make sure your Colab notebook is connected to a GPU:
1. Go to **Runtime** > **Change runtime type**
2. Hardware accelerator: **T4 GPU** (or any available GPU)
3. Click **Save**

In [ ]:
# Step 1: Verify Hardware Accelerator
import torch

print(f"PyTorch Version: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ GPU is NOT enabled! Go to Runtime -> Change runtime type -> Select T4 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU Detected: {device_name}")
print(f"✅ Total VRAM: {vram_gb:.2f} GB")
!nvidia-smi

In [ ]:
# Step 2: Install Required Libraries
!pip install -q --upgrade "transformers>=4.45.0" "peft>=0.12.0" "trl>=0.11.0" "bitsandbytes>=0.43.0" "accelerate>=0.34.0" "datasets>=2.20.0" "pyarrow<24"

In [ ]:
# Step 3: Clone Repository and Verify Dataset Files
import os
from pathlib import Path

repo_name = "Campus-Knowledge-Engine"
if not os.path.exists(repo_name):
    !git clone https://github.com/saicharan45264/Campus-Knowledge-Engine.git

DATA_DIR = Path(f"{repo_name}/fine_tuning") if os.path.exists(f"{repo_name}/fine_tuning") else Path(f"{repo_name}/training")
dataset_path = DATA_DIR / "campus_router_curated_v1.jsonl"
schema_path = DATA_DIR / "academic_tool_schemas.json"

if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

print(f"✅ Dataset verified: {dataset_path.resolve()}")
print(f"✅ Tool schemas verified: {schema_path.resolve()}")

In [ ]:
# Step 4: Model Selection & Quantization Settings
import torch
from transformers import BitsAndBytesConfig

# Recommended models for lightweight QLoRA (< 15 mins on Colab T4):
# 1. "Qwen/Qwen2.5-3B-Instruct" (Default, ungated, superb tool calling & argument extraction)
# 2. "Qwen/Qwen2.5-1.5B-Instruct" (Ultra-fast, ~3 min training)
# 3. "google/gemma-2-2b-it" (Requires HuggingFace login: huggingface-cli login)
# 4. "google/gemma-2-9b-it" (Fits in T4 16GB VRAM with 4-bit, ~15 min training)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Selected Base Model: {MODEL_ID}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
# Step 5: Load Curated Dataset & Prepare Chat Splits
import json

SYSTEM_INSTRUCTION = """Route the student's query to exactly one registered academic lookup tool. Invoke one tool from the list below, or make no tool call when the query is general or the current tool set cannot handle it. Extract only values stated or unambiguously implied; omit unknown optional arguments and never invent dates, course codes, student details, or facts.

Tools and arguments:
- lookup_academic_calendar: event_type (commencement, midterm, endsem, holiday, working_saturday, general), temporal_scope (next, all, specific, general), term, target_date, target_month, cohort, festival.
- lookup_class_timetable: query_type (venue, faculty, time_slot, day_schedule, lunch_break, general), course_code, weekday, slot_number, department, semester, class_name.
- lookup_elective_directory: elective_bucket (PE-IV, PE-V, PE-VI, free_elective, all).
- lookup_lab_schedule: lab_identifier, weekday, check_vacant_only, department, semester.
- lookup_exam_schedule: exam_type (end_sem, mid_term), course_code, department, semester. Use this for a course's exam date/session; use the calendar tool for semester-wide exam commencement.
- lookup_exam_seating: register_number, department, semester. For 'my seat' queries, leave register_number out; the signed-in profile supplies it.
- lookup_curriculum_concept: course_code_or_name, aspect (prerequisites, unit_syllabus, course_outcomes, blooms_taxonomy, credits_ltpc, general), unit_number.
- lookup_academic_regulations: topic, clause_number.
- lookup_past_exam_questions: course_code_or_name, topic, exam_type (mid_term, end_sem, all), marks.
For a general conversational query or a request for unsupported PYQ/CO mapping, do not call a tool so the application fallback can handle it."""

train_rows = []
eval_rows = []

with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("split") == "train":
            train_rows.append(row)
        else:
            eval_rows.append(row)

print(f"✅ Loaded {len(train_rows)} training samples and {len(eval_rows)} held-out evaluation samples.")

def format_example(row):
    query = str(row.get("query") or row.get("input") or "").strip()
    tool = row.get("tool")
    arguments = row.get("arguments") or {}
    assistant_content = json.dumps({"tool": tool, "arguments": arguments}, ensure_ascii=False)
    
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": query},
            {"role": "assistant", "content": assistant_content}
        ]
    }

train_dataset = [format_example(r) for r in train_rows]
eval_dataset = [format_example(r) for r in eval_rows]
print("Sample Chat Format:", json.dumps(train_dataset[0], indent=2))

In [ ]:
# Step 6: Initialize Tokenizer and 4-bit Base Model
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print("✅ Base model loaded into 4-bit VRAM successfully!")

In [ ]:
# Step 7: Prediction & Evaluation Utilities
import re

def extract_json_route(text):
    """Extract and parse the JSON tool call dictionary from model output."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            res = json.loads(match.group(0))
            if isinstance(res, dict) and ("tool" in res or "arguments" in res):
                return res.get("tool"), res.get("arguments") or {}
        except Exception:
            pass
    return None, {}

def predict_single_query(m, tok, query_text):
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": query_text},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    
    with torch.inference_mode():
        out = m.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
        )
    
    output_text = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    pred_tool, pred_args = extract_json_route(output_text)
    return pred_tool, pred_args, output_text

def score_dataset(m, tok, rows, max_samples=30):
    test_set = rows[:max_samples]
    tool_correct = 0
    exact_correct = 0
    
    print(f"Evaluating on {len(test_set)} held-out queries...")
    for item in test_set:
        q = item.get("query") or item.get("input")
        exp_tool = item.get("tool")
        exp_args = item.get("arguments") or {}
        
        p_tool, p_args, _ = predict_single_query(m, tok, q)
        if p_tool == exp_tool:
            tool_correct += 1
            if p_args == exp_args:
                exact_correct += 1
                
    tool_acc = (tool_correct / len(test_set)) * 100
    exact_acc = (exact_correct / len(test_set)) * 100
    print(f"  → Tool Accuracy: {tool_acc:.1f}% ({tool_correct}/{len(test_set)})")
    print(f"  → Exact Tool & Args Match: {exact_acc:.1f}% ({exact_correct}/{len(test_set)})")
    return tool_acc, exact_acc

In [ ]:
# Step 8: Benchmark Base Model (Zero-Shot Baseline)
print("=== Base Model Evaluation (Pre-Training Baseline) ===")
base_tool_acc, base_exact_acc = score_dataset(model, tokenizer, eval_rows, max_samples=25)

In [ ]:
# Step 9: Configure QLoRA and SFTTrainer
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

train_ds = Dataset.from_list(train_dataset)
eval_ds = Dataset.from_list(eval_dataset)

training_args = SFTConfig(
    output_dir="./router_training_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    max_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)
print("✅ QLoRA & SFTTrainer configured. Ready to fine-tune!")

In [ ]:
# Step 10: Run QLoRA Fine-Tuning (Under 10 minutes on T4 GPU)
import time

t_start = time.time()
print("🚀 Training started on GPU...")

train_result = trainer.train()

duration_min = (time.time() - t_start) / 60
print(f"🎉 Fine-tuning finished in {duration_min:.2f} minutes!")

In [ ]:
# Step 11: Benchmark Fine-Tuned Model & Display Comparison
print("=== Fine-Tuned Model Evaluation (Post-Training) ===")
tuned_tool_acc, tuned_exact_acc = score_dataset(trainer.model, tokenizer, eval_rows, max_samples=25)

print("\n" + "="*55)
print("🎯 ACCURACY BENCHMARK COMPARISON:")
print(f"  Tool Selection Accuracy:      {base_tool_acc:.1f}%  ──▶  {tuned_tool_acc:.1f}%")
print(f"  Exact Tool & Args Match:      {base_exact_acc:.1f}%  ──▶  {tuned_exact_acc:.1f}%")
print("="*55)

In [ ]:
# Step 12: Interactive Query Demonstration
# 0. Uninstall conflicting torchao and free GPU memory
!pip uninstall -y torchao

import gc
import sys
import os
import zipfile
import json
import re
import torch

# Free any dangling GPU memory
for var in ['base_model', 'eval_model', 'trainer', 'base_m', 'model']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

# Neutralize torchao import checks in PEFT
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Extract adapter if zip was uploaded
if os.path.exists("mach_router_lora_adapter.zip") and not os.path.exists("./mach_router_lora_adapter/adapter_model.safetensors"):
    with zipfile.ZipFile("mach_router_lora_adapter.zip", "r") as z:
        z.extractall("./mach_router_lora_adapter")

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "./mach_router_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading 4-bit base model and attaching fine-tuned adapter...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

eval_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
eval_model.eval()
print("✅ Fine-tuned adapter attached successfully!\n")

SYSTEM_INSTRUCTION = """Route the student's query to exactly one registered academic lookup tool. Invoke one tool from the list below, or make no tool call when the query is general or the current tool set cannot handle it. Extract only values stated or unambiguously implied; omit unknown optional arguments and never invent dates, course codes, student details, or facts.

Tools and arguments:
- lookup_academic_calendar: event_type (commencement, midterm, endsem, holiday, working_saturday, general), temporal_scope (next, all, specific, general), term, target_date, target_month, cohort, festival.
- lookup_class_timetable: query_type (venue, faculty, time_slot, day_schedule, lunch_break, general), course_code, weekday, slot_number, department, semester, class_name.
- lookup_elective_directory: elective_bucket (PE-IV, PE-V, PE-VI, free_elective, all).
- lookup_lab_schedule: lab_identifier, weekday, check_vacant_only, department, semester.
- lookup_exam_schedule: exam_type (end_sem, mid_term), course_code, department, semester. Use this for a course's exam date/session; use the calendar tool for semester-wide exam commencement.
- lookup_exam_seating: register_number, department, semester. For 'my seat' queries, leave register_number out; the signed-in profile supplies it.
- lookup_curriculum_concept: course_code_or_name, aspect (prerequisites, unit_syllabus, course_outcomes, blooms_taxonomy, credits_ltpc, general), unit_number.
- lookup_academic_regulations: topic, clause_number.
- lookup_past_exam_questions: course_code_or_name, topic, exam_type (mid_term, end_sem, all), marks.
For a general conversational query or a request for unsupported PYQ/CO mapping, do not call a tool so the application fallback can handle it."""

def extract_json_route(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            res = json.loads(match.group(0))
            if isinstance(res, dict) and ("tool" in res or "arguments" in res):
                return res.get("tool"), res.get("arguments") or {}
        except Exception:
            pass
    return None, {}

def predict_single_query(model, tokenizer, query_text):
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": query_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.01,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    tool, args = extract_json_route(response_text)
    return tool, args, response_text

test_queries = [
    "When do mid-term examinations commence for the 3rd semester?",
    "Where is the 23CSE204 class scheduled on Thursday morning?",
    "Can I see 8 mark questions on ER modeling from previous exams?",
    "Is D-102 lab vacant on Friday afternoon?",
    "What are the prerequisites for 23CSE301?",
    "Can you write a poem about the campus library?",
    "Where is my exam seat for roll number CB.EN.U4CSE21001?"
]

print("🧪 Live Prediction Demo with Fine-Tuned Model:\n")
for query in test_queries:
    p_tool, p_args, _ = predict_single_query(eval_model, tokenizer, query)
    print(f"Query: \"{query}\"")
    print(f"Predicted Call: {json.dumps({'tool': p_tool, 'arguments': p_args}, indent=2)}\n")

In [ ]:
# Step 13: Export LoRA Adapter and Download Bundle
import shutil
from pathlib import Path

export_dir = Path("./mach_router_lora_adapter")
export_dir.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter weights and tokenizer configuration
trainer.model.save_pretrained(str(export_dir))
tokenizer.save_pretrained(str(export_dir))

# Save training manifest
manifest = {
    "base_model": MODEL_ID,
    "epochs": 3,
    "training_samples": len(train_rows),
    "held_out_samples": len(eval_rows),
    "training_time_minutes": round(duration_min, 2),
    "base_tool_accuracy": base_tool_acc,
    "fine_tuned_tool_accuracy": tuned_tool_acc,
    "base_exact_accuracy": base_exact_acc,
    "fine_tuned_exact_accuracy": tuned_exact_acc,
}
(export_dir / "training_manifest.json").write_text(json.dumps(manifest, indent=2))

# Compress to zip archive
zip_filename = "mach_router_lora_adapter"
zip_path = shutil.make_archive(zip_filename, "zip", root_dir=str(export_dir))
print(f"✅ LoRA Adapter bundled at: {zip_path}")

# Trigger browser download if inside Colab
try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
    print("⬇️ Download prompt opened in your browser!")
except Exception:
    print("To download manually, right click on mach_router_lora_adapter.zip in the Colab file tree.")